<h1><b>DATA</b></h1>

In [1]:
library(dplyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
data_transformed

ERROR: Error: object 'data_transformed' not found


In [3]:
set.seed(427)

# Создаем новые данные с большим количеством наблюдений
data <- read.csv("diabetes.csv")

prepare_data <- function(data) {
  data <- data %>%
    mutate(
      Gender = ifelse(Gender == "Male", 1, 0),
      Blood.Pressure = ifelse(Blood.Pressure == "High", 1, 0),
      Family.History.of.Diabetes = ifelse(Family.History.of.Diabetes == "Yes", 1, 0),
      Smoking = ifelse(Smoking == "Yes", 1, 0),
      Diet = ifelse(Diet == "Healthy", 1, 0),
      Exercise = ifelse(Exercise == "Regular", 1, 0),
      Diagnosis = ifelse(Diagnosis == "Yes", 1, 0)
    )
  
  return(data)
}

data_transformed = prepare_data(data)

# Подготовка данных
X <- as.matrix(data_transformed[, -ncol(data_transformed)])  # исключаем последний столбец (Success)
y <- as.vector(data_transformed$Diagnosis)  # целевая переменная (успех или неуспех)

# Разделяем данные на обучающую и тестовую выборки
set.seed(427)
train_indices <- sample(1:nrow(data_transformed), size = 0.8 * nrow(data_transformed))

# Обучающая выборка
X_train <- X[train_indices, ]
y_train <- y[train_indices]

# Тестовая выборка
X_test <- X[-train_indices, ]
y_test <- y[-train_indices]

# Выводим пример обучающего набора данных
head(X_train)
head(y_train)


Age,Gender,BMI,Blood.Pressure,FBS,HbA1c,Family.History.of.Diabetes,Smoking,Diet,Exercise
55,0,30,1,120,6.4,1,1,0,0
20,0,19,0,100,5.7,1,1,0,0
57,0,54,1,260,11.3,0,0,1,1
65,1,35,1,140,7.1,1,1,0,0
43,0,42,1,180,8.5,1,1,0,0
33,0,32,1,140,7.1,0,1,0,0


[1] 1 0 0 1 0 1

<h1><b>TREE + ADA BOOST v1</b></h1>

In [4]:
#install.packages("R6")
library(R6)
# Определяем класс Node
Node <- R6Class("Node",
  public = list(
    feature = NULL,
    threshold = NULL,
    left = NULL,
    right = NULL,
    value = NULL,
    
    initialize = function(feature = NULL, threshold = NULL, left = NULL, right = NULL, value = NULL) {
      self$feature <- feature
      self$threshold <- threshold
      self$left <- left
      self$right <- right
      self$value <- value
    },
    
    is_leaf_node = function() {
      return(!is.null(self$value))
    }
  )
)

# Определяем класс DecisionTree
DecisionTree <- R6Class("DecisionTree",
  public = list(
    min_samples = 2,
    max_depth = 100,
    tree = NULL,
    
    initialize = function(min_samples = 2, max_depth = 10) {
      self$min_samples <- min_samples
      self$max_depth <- max_depth
    },
    
    fit = function(X, y) {
      self$tree <- self$grow_tree(X, y)
    },
    
    predict = function(X) {
      sapply(1:nrow(X), function(i) self$traverse_tree(X[i, ], self$tree))
    },
    
    most_common = function(y) {
      return(names(sort(table(y), decreasing = TRUE)[1]))
    },
    
    entropy = function(y) {
      prob <- table(y) / length(y)
      -sum(prob * log2(prob))
    },
    
    best_split = function(X, y) {
      best_gain <- -Inf
      best_feature <- NULL
      best_threshold <- NULL
      for (i in 1:ncol(X)) {
        thresholds <- unique(X[, i])
        for (threshold in thresholds) {
          gain <- self$information_gain(X[, i], y, threshold)
          if (gain > best_gain) {
            best_gain <- gain
            best_feature <- i
            best_threshold <- threshold
          }
        }
      }
      return(list(best_feature = best_feature, best_threshold = best_threshold))
    },
    
    information_gain = function(X_column, y, threshold) {
      n <- length(y)
      parent_entropy <- self$entropy(y)
      
      left_indices <- which(X_column <= threshold)
      right_indices <- which(X_column > threshold)
      
      left_entropy <- self$entropy(y[left_indices])
      right_entropy <- self$entropy(y[right_indices])
      
      left_weight <- length(left_indices) / n
      right_weight <- length(right_indices) / n
      
      child_entropy <- left_weight * left_entropy + right_weight * right_entropy
      return(parent_entropy - child_entropy)
    },
    
    grow_tree = function(X, y, depth = 0) {
      n_samples <- nrow(X)
      n_labels <- length(unique(y))
      
      if (n_samples < self$min_samples || n_labels == 1 || depth >= self$max_depth) {
        return(Node$new(value = self$most_common(y)))
      }
      
      split <- self$best_split(X, y)
      best_feature <- split$best_feature
      best_threshold <- split$best_threshold
      
      left_indices <- which(X[, best_feature] <= best_threshold)
      right_indices <- which(X[, best_feature] > best_threshold)
      
      left_node <- self$grow_tree(X[left_indices, ], y[left_indices], depth + 1)
      right_node <- self$grow_tree(X[right_indices, ], y[right_indices], depth + 1)
      
      return(Node$new(feature = best_feature, threshold = best_threshold, left = left_node, right = right_node))
    },
    
    traverse_tree = function(x, tree) {
      if (tree$is_leaf_node()) {
        return(tree$value)
      }
      
      if (x[tree$feature] <= tree$threshold) {
        return(self$traverse_tree(x, tree$left))
      } else {
        return(self$traverse_tree(x, tree$right))
      }
    }
  )
)



In [5]:
# Создаем объект дерева
tree <- DecisionTree$new(max_depth = 1)

# Обучаем дерево на тренировочных данных
tree$fit(X_train, y_train)

# Делаем предсказания на тестовых данных
predictions <- tree$predict(X_test)

# Посмотрим на результаты предсказаний
print(predictions)

# Оценим точность модели на тестовых данных
accuracy <- mean(predictions == y_test)
print(paste("Accuracy:", accuracy))


 [1] "0" "0" "0" "0" "0" "0" "0" "0" "0" "0" "0" "0" "0" "0" "0" "0" "0" "0" "0"
[20] "0" "0" "0" "0" "0" "0" "0"
[1] "Accuracy: 0.692307692307692"


In [6]:
AdaBoost <- R6Class("AdaBoost",
  public = list(
    n_estimators = 10,
    trees = list(),
    asay = list(),
    weights = list(),
    
    initialize = function(n_estimators = 10) {
      self$n_estimators <- n_estimators
    },
    
    fit = function(X, y) {
      n <- length(y)
      w <- rep(1 / n, n)
      
      for (i in 1:self$n_estimators) {
        tree <- DecisionTree$new(max_depth = 5)
        tree$fit(X, y)
        self$trees[[i]] <- tree
        
        labels <- tree$predict(X)
        missed <- as.integer(labels != y)
        
        total_error <- sum(w * missed)
        amount_of_say <- 0.5 * log((1 - total_error) / (total_error + 1e-7))
        
        missed <- missed * 2 - 1
        w <- w * exp(amount_of_say * missed)
        w <- w / sum(w)
        
        self$asay[[i]] <- amount_of_say
        self$weights[[i]] <- w
      }
    },
    
    predict = function(X) {
      predictions <- rep(0, nrow(X))
      
      for (i in 1:self$n_estimators) {
        tree_predictions <- self$trees[[i]]$predict(X)
        tree_predictions <- as.numeric(tree_predictions) * 2 - 1  # Преобразуем в числа 1 или -1
        predictions <- predictions + (tree_predictions * self$asay[[i]])
      }
      
      return(ifelse(predictions <= 0, 0, 1))
    },
    
    score = function(predicted, y) {
      return(mean(predicted == y))
    }
  )
)


In [7]:
# Обучаем и проверяем AdaBoost
ab_clf <- AdaBoost$new(n_estimators = 30)
ab_clf$fit(X_train, y_train)
predicted <- ab_clf$predict(X_test)

# Оценка модели
print(ab_clf$score(predicted, y_test))

[1] 0.9615385


<h1><b>PREDICT NEW DATA</b></h1>

In [12]:
str(data)

'data.frame':	128 obs. of  11 variables:
 $ Age                       : int  45 55 65 75 40 50 60 70 45 55 ...
 $ Gender                    : chr  "Male" "Female" "Male" "Female" ...
 $ BMI                       : int  25 30 35 40 20 25 30 35 25 30 ...
 $ Blood.Pressure            : chr  "Normal" "High" "High" "High" ...
 $ FBS                       : int  100 120 140 160 80 100 120 140 80 100 ...
 $ HbA1c                     : num  5.7 6.4 7.1 7.8 5 5.7 6.4 7.1 5 5.7 ...
 $ Family.History.of.Diabetes: chr  "No" "Yes" "Yes" "Yes" ...
 $ Smoking                   : chr  "No" "Yes" "Yes" "Yes" ...
 $ Diet                      : chr  "Healthy" "Poor" "Poor" "Poor" ...
 $ Exercise                  : chr  "Regular" "No" "No" "No" ...
 $ Diagnosis                 : chr  "No" "Yes" "Yes" "Yes" ...


In [23]:
prepare_new_data <- function(data) {
  data <- data %>%
    mutate(
      Gender = ifelse(Gender == "Male", 1, 0),
      Blood.Pressure = ifelse(Blood.Pressure == "High", 1, 0),
      Family.History.of.Diabetes = ifelse(Family.History.of.Diabetes == "Yes", 1, 0),
      Smoking = ifelse(Smoking == "Yes", 1, 0),
      Diet = ifelse(Diet == "Healthy", 1, 0),
      Exercise = ifelse(Exercise == "Regular", 1, 0),
    )
  
  return(data)
}

age <- 50
gender <- "Male"
bmi <- 28
blood_pressure <- "High"
fbs <- 110
hba1c <- 5.9
family_history <- "No"
smoking <- "No"
diet <- "Healthy"
exercise <- "Regular"

new_data <- data.frame(
    Age = as.integer(age),
    Gender = as.character(gender),
    BMI = as.integer(bmi),
    Blood.Pressure = as.character(blood_pressure),
    FBS = as.integer(fbs),
    HbA1c = as.numeric(hba1c),
    Family.History.of.Diabetes = as.character(family_history),
    Smoking = as.character(smoking),
    Diet = as.character(diet),
    Exercise = as.character(exercise)
  )

In [24]:
new_data

Age,Gender,BMI,Blood.Pressure,FBS,HbA1c,Family.History.of.Diabetes,Smoking,Diet,Exercise
<int>,<chr>,<int>,<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>
50,Male,28,High,110,5.9,No,No,Healthy,Regular


In [25]:
new_data_transformed = prepare_new_data(new_data)

In [29]:
str(new_data_transformed)

'data.frame':	1 obs. of  10 variables:
 $ Age                       : int 50
 $ Gender                    : num 1
 $ BMI                       : int 28
 $ Blood.Pressure            : num 1
 $ FBS                       : int 110
 $ HbA1c                     : num 5.9
 $ Family.History.of.Diabetes: num 0
 $ Smoking                   : num 0
 $ Diet                      : num 1
 $ Exercise                  : num 1


In [27]:
# Предсказание для новых данных
prediction <- ab_clf$predict(new_data_transformed)

# Вывод результата (0 - нет диабета, 1 - есть диабет)
print(prediction)


[1] 0
